# Experiment 10: Architecture Exploration

## 1. Research Question
Can alternative neural network architectures (e.g., increased capacity, convolutions, residual connections, or multi-task joint learning) improve the estimation of CBF and ATT compared to the baseline fully connected DNN? Importantly, do architectural changes affect the 6-PLD and 3-PLD models differently, and can they reduce the 3-PLD performance gap?

## 2. Experimental Controls
- **Data:** Uniformly sampled CBF [0, 100] and ATT [0.5, 3.0s]. 100,000 train, 10,000 val, 10,000 test.
- **Reference Noise:** `reference_cbf = 50.0`, `reference_att_s = 1.6`. SNR = 10.0.
- **Configurations:** 6-PLD Baseline `[1.525, 2.025, 2.525, 3.025, 3.525, 4.025]` and 3-PLD Selected `[1.525, 2.525, 3.025]`.
- **Metrics:** MAE, RMSE, R2, Bias. Focus on RMSE and 3-PLD vs 6-PLD relative degradation.


In [1]:
import sys, os, copy, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from IPython.display import display, Markdown
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score

project_root = Path.cwd().parent if not (Path.cwd() / "src").exists() else Path.cwd()
sys.path.insert(0, str(project_root / "src"))
from simulation import SimulationConfig, generate_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

out_root = project_root / "results" / "architecture_experiments"
plot_dir = project_root / "figures" / "architecture_experiments"
out_root.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)

# Training Constants
EXPERIMENT_CONFIG = {
    "n_train": 100000,
    "n_val": 10000,
    "n_test": 10000,
    "train_seed": 42 + 10,
    "val_seed": 42 + 20,
    "test_seed": 999,
    "snr": 10.0,
    "batch_size": 512,
    "lr": 1e-3,
    "patience": 20,
    "max_epochs": 150,
    "cbf_width": 50,
    "att_width": 100,
    "grad_clip": 1.0
}

SELECTED_3PLD_INDICES = [0, 2, 3]

# Generate Shared Dataset
cfg = SimulationConfig(reference_cbf=50.0, reference_att_s=1.6)
X_tr_full, Y_tr, _ = generate_dataset(EXPERIMENT_CONFIG["n_train"], cfg, EXPERIMENT_CONFIG["train_seed"], snr=EXPERIMENT_CONFIG["snr"])
X_va_full, Y_va, _ = generate_dataset(EXPERIMENT_CONFIG["n_val"], cfg, EXPERIMENT_CONFIG["val_seed"], snr=EXPERIMENT_CONFIG["snr"])
X_te_full, Y_te, _ = generate_dataset(EXPERIMENT_CONFIG["n_test"], cfg, EXPERIMENT_CONFIG["test_seed"], snr=EXPERIMENT_CONFIG["snr"])

CBF_mean, CBF_std = np.mean(Y_tr[:, 0]), np.std(Y_tr[:, 0])
ATT_mean, ATT_std = np.mean(Y_tr[:, 1]), np.std(Y_tr[:, 1])

Y_tr_norm_cbf = ((Y_tr[:, 0] - CBF_mean) / CBF_std).astype('float32')
Y_va_norm_cbf = ((Y_va[:, 0] - CBF_mean) / CBF_std).astype('float32')
Y_tr_norm_att = ((Y_tr[:, 1] - ATT_mean) / ATT_std).astype('float32')
Y_va_norm_att = ((Y_va[:, 1] - ATT_mean) / ATT_std).astype('float32')

Y_tr_norm_both = np.column_stack((Y_tr_norm_cbf, Y_tr_norm_att))
Y_va_norm_both = np.column_stack((Y_va_norm_cbf, Y_va_norm_att))


Device: cuda


In [2]:
# 1. Baseline MLP (Current Architecture)
class BaselineMLP(nn.Module):
    def __init__(self, input_dim, width):
        super().__init__()
        layers = [nn.Linear(input_dim, width), nn.ELU()]
        for _ in range(8):
            layers += [nn.Linear(width, width), nn.ELU()]
        layers.append(nn.Linear(width, 1))
        self.backbone = nn.Sequential(*layers)
        for layer in self.backbone:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)
    def forward(self, x):
        return self.backbone(x)

# 2. Wider MLP (Double Capacity)
class WiderMLP(nn.Module):
    def __init__(self, input_dim, width):
        super().__init__()
        width = width * 2 # Double width
        layers = [nn.Linear(input_dim, width), nn.ELU()]
        for _ in range(8):
            layers += [nn.Linear(width, width), nn.ELU()]
        layers.append(nn.Linear(width, 1))
        self.backbone = nn.Sequential(*layers)
    def forward(self, x): return self.backbone(x)

# 3. Residual MLP
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.layer = nn.Sequential(nn.Linear(dim, dim), nn.ELU(), nn.Linear(dim, dim), nn.ELU())
    def forward(self, x): return x + self.layer(x)

class ResidualMLP(nn.Module):
    def __init__(self, input_dim, width):
        super().__init__()
        self.in_layer = nn.Sequential(nn.Linear(input_dim, width), nn.ELU())
        self.blocks = nn.Sequential(*[ResidualBlock(width) for _ in range(4)])
        self.out_layer = nn.Linear(width, 1)
    def forward(self, x):
        return self.out_layer(self.blocks(self.in_layer(x)))

# 4. 1D CNN
class Conv1DNet(nn.Module):
    def __init__(self, seq_len, width):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, padding='same'), nn.ELU(),
            nn.Conv1d(16, 32, kernel_size=3, padding='same'), nn.ELU()
        )
        self.fc = nn.Sequential(nn.Linear(32 * seq_len, width), nn.ELU(), nn.Linear(width, 1))
    def forward(self, x):
        return self.fc(self.conv(x.unsqueeze(1)).view(x.shape[0], -1))

# 5. Multi-Task MLP
class MultiTaskMLP(nn.Module):
    def __init__(self, input_dim, width):
        super().__init__()
        layers = [nn.Linear(input_dim, width), nn.ELU()]
        for _ in range(6): layers += [nn.Linear(width, width), nn.ELU()]
        self.shared = nn.Sequential(*layers)
        self.cbf = nn.Sequential(nn.Linear(width, width//2), nn.ELU(), nn.Linear(width//2, 1))
        self.att = nn.Sequential(nn.Linear(width, width//2), nn.ELU(), nn.Linear(width//2, 1))
    def forward(self, x):
        f = self.shared(x)
        return torch.cat([self.cbf(f), self.att(f)], dim=1)


In [3]:
def calc_metrics(y_true, y_pred):
    e = y_pred - y_true
    return {
        "MAE": float(np.mean(np.abs(e))),
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "R2": float(r2_score(y_true, y_pred)),
        "Bias": float(np.mean(e)),
    }

def train_generic(net, X_tr, Y_tr, X_va, Y_va):
    if len(Y_tr.shape) == 1: Y_tr = Y_tr[:, None]
    if len(Y_va.shape) == 1: Y_va = Y_va[:, None]
    
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(Y_tr)), batch_size=EXPERIMENT_CONFIG["batch_size"], shuffle=True)
    val_loader = DataLoader(TensorDataset(torch.from_numpy(X_va), torch.from_numpy(Y_va)), batch_size=EXPERIMENT_CONFIG["batch_size"])
    
    opt = torch.optim.Adam(net.parameters(), lr=EXPERIMENT_CONFIG["lr"])
    loss_fn = nn.L1Loss()
    best_val = float('inf')
    best_state = copy.deepcopy(net.state_dict())
    stale = 0
    hist = []
    
    for epoch in range(1, EXPERIMENT_CONFIG["max_epochs"] + 1):
        net.train()
        train_loss = 0.0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(net(xb.to(device)), yb.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), EXPERIMENT_CONFIG["grad_clip"])
            opt.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)
        
        net.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                val_loss += loss_fn(net(xb.to(device)), yb.to(device)).item()
        val_loss /= len(val_loader)
        
        hist.append({"epoch": epoch, "train_mae": train_loss, "validation_mae": val_loss})
        
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(net.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= EXPERIMENT_CONFIG["patience"]:
                break
                
    net.load_state_dict(best_state)
    return pd.DataFrame(hist)


In [4]:
architectures = {
    "BaselineMLP": {"class": BaselineMLP, "type": "single"},
    "WiderMLP": {"class": WiderMLP, "type": "single"},
    "ResidualMLP": {"class": ResidualMLP, "type": "single"},
    "Conv1DNet": {"class": Conv1DNet, "type": "single"},
    "MultiTaskMLP": {"class": MultiTaskMLP, "type": "multi"}
}

results = []
summary_file = out_root / "architecture_summary.csv"

# Optional Resume
if summary_file.exists():
    results = pd.read_csv(summary_file).to_dict('records')

for arch_name, info in architectures.items():
    for n_plds, indices, pld_name in [(6, list(range(6)), "6-PLD"), (3, SELECTED_3PLD_INDICES, "3-PLD")]:
        
        if any(r["architecture"] == arch_name and r["pld_configuration"] == pld_name for r in results):
            print(f"Skipping {arch_name} {pld_name} (already trained)")
            continue
            
        print(f"\n--- Training {arch_name} [{pld_name}] ---")
        
        X_tr = X_tr_full[:, indices]
        X_va = X_va_full[:, indices]
        X_te = X_te_full[:, indices]
        
        X_mean = X_tr.mean(0, keepdims=True).astype('float32')
        X_std = X_tr.std(0, keepdims=True).astype('float32') + 1e-8
        
        X_tr_n = ((X_tr - X_mean) / X_std).astype('float32')
        X_va_n = ((X_va - X_mean) / X_std).astype('float32')
        X_te_n = ((X_te - X_mean) / X_std).astype('float32')
        
        xt = torch.from_numpy(X_te_n).to(device)
        
        start_time = time.time()
        
        if info["type"] == "single":
            # Train CBF
            cbf_net = info["class"](n_plds, EXPERIMENT_CONFIG["cbf_width"]).to(device)
            cbf_hist = train_generic(cbf_net, X_tr_n, Y_tr_norm_cbf, X_va_n, Y_va_norm_cbf)
            
            # Train ATT
            att_net = info["class"](n_plds, EXPERIMENT_CONFIG["att_width"]).to(device)
            att_hist = train_generic(att_net, X_tr_n, Y_tr_norm_att, X_va_n, Y_va_norm_att)
            
            # Eval
            cbf_net.eval(); att_net.eval()
            with torch.no_grad():
                cbf_pred = cbf_net(xt).cpu().squeeze(1).numpy() * CBF_std + CBF_mean
                att_pred = att_net(xt).cpu().squeeze(1).numpy() * ATT_std + ATT_mean
                
            best_cbf_ep = int(cbf_hist["validation_mae"].idxmin())
            best_att_ep = int(att_hist["validation_mae"].idxmin())
            
        else:
            # Multi-task
            # Use att_width (100) for shared capacity
            net = info["class"](n_plds, EXPERIMENT_CONFIG["att_width"]).to(device)
            hist = train_generic(net, X_tr_n, Y_tr_norm_both, X_va_n, Y_va_norm_both)
            
            net.eval()
            with torch.no_grad():
                preds = net(xt).cpu().numpy()
                cbf_pred = preds[:, 0] * CBF_std + CBF_mean
                att_pred = preds[:, 1] * ATT_std + ATT_mean
                
            best_cbf_ep = int(hist["validation_mae"].idxmin())
            best_att_ep = best_cbf_ep
            
        dur = time.time() - start_time
        
        cbf_pred = np.clip(cbf_pred, 0.0, 100.0)
        att_pred = np.clip(att_pred, 0.5, 3.0)
        
        m_cbf = calc_metrics(Y_te[:, 0], cbf_pred)
        m_att = calc_metrics(Y_te[:, 1], att_pred)
        
        row = {
            "architecture": arch_name,
            "variant": info["type"],
            "n_plds": n_plds,
            "pld_configuration": pld_name,
            "train_samples": EXPERIMENT_CONFIG["n_train"],
            "cbf_mae": m_cbf["MAE"],
            "cbf_rmse": m_cbf["RMSE"],
            "cbf_r2": m_cbf["R2"],
            "cbf_bias": m_cbf["Bias"],
            "att_mae": m_att["MAE"],
            "att_rmse": m_att["RMSE"],
            "att_r2": m_att["R2"],
            "att_bias": m_att["Bias"],
            "best_cbf_epoch": best_cbf_ep,
            "best_att_epoch": best_att_ep,
            "train_duration_s": dur,
            "seed": EXPERIMENT_CONFIG["train_seed"]
        }
        results.append(row)
        pd.DataFrame(results).to_csv(summary_file, index=False)
        
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

display(Markdown("### Architecture Screening Completed"))



--- Training BaselineMLP [6-PLD] ---



--- Training BaselineMLP [3-PLD] ---



--- Training WiderMLP [6-PLD] ---



--- Training WiderMLP [3-PLD] ---



--- Training ResidualMLP [6-PLD] ---


...


--- Training Conv1DNet [6-PLD] ---



--- Training Conv1DNet [3-PLD] ---



--- Training MultiTaskMLP [6-PLD] ---



--- Training MultiTaskMLP [3-PLD] ---


### Architecture Screening Completed

In [5]:
df = pd.read_csv(summary_file)
comp_records = []

for arch in df['architecture'].unique():
    sub = df[df['architecture'] == arch]
    if len(sub) == 2:
        r6 = sub[sub['n_plds'] == 6].iloc[0]
        r3 = sub[sub['n_plds'] == 3].iloc[0]
        
        comp_records.append({
            "architecture": arch,
            "6PLD_CBF_RMSE": r6['cbf_rmse'],
            "3PLD_CBF_RMSE": r3['cbf_rmse'],
            "CBF_RMSE_Degradation_%": (r3['cbf_rmse'] - r6['cbf_rmse']) / r6['cbf_rmse'] * 100,
            "6PLD_ATT_RMSE": r6['att_rmse'],
            "3PLD_ATT_RMSE": r3['att_rmse'],
            "ATT_RMSE_Degradation_%": (r3['att_rmse'] - r6['att_rmse']) / r6['att_rmse'] * 100
        })

df_comp = pd.DataFrame(comp_records)
df_comp.to_csv(out_root / "architecture_comparison.csv", index=False)
display(Markdown("### Relative Performance by Architecture"))
display(df_comp)


### Relative Performance by Architecture

,architecture,6PLD_CBF_RMSE,3PLD_CBF_RMSE,CBF_RMSE_Degradation_%,6PLD_ATT_RMSE,3PLD_ATT_RMSE,ATT_RMSE_Degradation_%
0,BaselineMLP,4.431816,4.934339,11.338974,0.363350,0.379656,4.487609
1,WiderMLP,4.370737,4.922171,12.616501,0.367862,0.382362,3.941552
2,ResidualMLP,4.380456,4.905046,11.975693,0.363345,0.379248,4.376853
3,Conv1DNet,4.417591,4.928883,11.574000,0.363775,0.382341,5.103597
4,MultiTaskMLP,4.443591,4.928135,10.904340,0.362543,0.380168,4.861557


In [6]:
# Visualizations
df = pd.read_csv(summary_file)
df_comp = pd.read_csv(out_root / "architecture_comparison.csv")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. 6-PLD CBF
ax = axes[0,0]
sub = df[df['n_plds']==6].sort_values('cbf_rmse')
ax.bar(sub['architecture'], sub['cbf_rmse'], color='royalblue')
ax.set_title("6-PLD CBF RMSE by Architecture")
ax.set_ylabel("CBF RMSE")
ax.tick_params(axis='x', rotation=45)

# 2. 6-PLD ATT
ax = axes[0,1]
sub = df[df['n_plds']==6].sort_values('att_rmse')
ax.bar(sub['architecture'], sub['att_rmse'], color='seagreen')
ax.set_title("6-PLD ATT RMSE by Architecture")
ax.set_ylabel("ATT RMSE")
ax.tick_params(axis='x', rotation=45)

# 3. 3-PLD CBF
ax = axes[1,0]
sub = df[df['n_plds']==3].sort_values('cbf_rmse')
ax.bar(sub['architecture'], sub['cbf_rmse'], color='coral')
ax.set_title("3-PLD CBF RMSE by Architecture")
ax.set_ylabel("CBF RMSE")
ax.tick_params(axis='x', rotation=45)

# 4. 3-PLD ATT
ax = axes[1,1]
sub = df[df['n_plds']==3].sort_values('att_rmse')
ax.bar(sub['architecture'], sub['att_rmse'], color='mediumorchid')
ax.set_title("3-PLD ATT RMSE by Architecture")
ax.set_ylabel("ATT RMSE")
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(plot_dir / "01_RMSE_Bar_Charts.png", dpi=200)
plt.close(fig)


# 5. Relative Degradation Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
df_comp_sorted_cbf = df_comp.sort_values('CBF_RMSE_Degradation_%')
ax1.bar(df_comp_sorted_cbf['architecture'], df_comp_sorted_cbf['CBF_RMSE_Degradation_%'], color='darkorange')
ax1.set_title("CBF RMSE Degradation (6-PLD -> 3-PLD)")
ax1.set_ylabel("Degradation (%)")
ax1.tick_params(axis='x', rotation=45)

df_comp_sorted_att = df_comp.sort_values('ATT_RMSE_Degradation_%')
ax2.bar(df_comp_sorted_att['architecture'], df_comp_sorted_att['ATT_RMSE_Degradation_%'], color='purple')
ax2.set_title("ATT RMSE Degradation (6-PLD -> 3-PLD)")
ax2.set_ylabel("Degradation (%)")
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
fig.savefig(plot_dir / "02_Relative_Degradation.png", dpi=200)
plt.close(fig)

display(Markdown("![RMSE Bars](../figures/architecture_experiments/01_RMSE_Bar_Charts.png)"))
display(Markdown("![Degradation Bars](../figures/architecture_experiments/02_Relative_Degradation.png)"))


![RMSE Bars](../figures/architecture_experiments/01_RMSE_Bar_Charts.png)

![Degradation Bars](../figures/architecture_experiments/02_Relative_Degradation.png)

## Analysis and Conclusions
The automated scripts have generated the metrics across all 5 Stage-1 & Stage-2 architectures. 
Review the output CSVs and graphs to determine if any alternative architecture reliably outperformed the standard BaselineMLP, or if the multi-task representation closed the performance gap for the 3-PLD configuration.
